In [31]:
import numpy as np
import h5py
from astropy.cosmology import FlatLambdaCDM
import astropy.units as u
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)

In [32]:
print(cosmo.kpc_proper_per_arcmin(0.5))
print(cosmo.kpc_comoving_per_arcmin(0.5))

366.2525721757739 kpc / arcmin
549.3788582636608 kpc / arcmin


In [30]:
def cal_log10_sigma_star(logmstar,re_arcsec,zd) -> float:
    """Calculate the stellar velocity dispersion using the virial theorem.

    Parameters
    ----------
    logmstar : float
        The logarithm of the stellar mass in solar masses.
    re_arcsec : float
        The effective radius in arcseconds.
    zd : float
        The redshift of the lens galaxy.

    Returns
    -------
    sigma_star : float
        The stellar velocity dispersion in km/s.
    """
    # Convert logmstar to mstar in solar masses
    mstar = 10 ** logmstar *u.Msun

    
    # Convert re from arcseconds to kpc
    re_kpc = re_arcsec *u.arcsec * cosmo.kpc_proper_per_arcmin(zd)
    # re_kpc = re_kpc.value
    # Calculate Sigma_star
    sigma_star = mstar / (2 * np.pi * re_kpc ** 2)
    
    return np.log10(sigma_star.to(u.Msun / u.kpc**2).value)


In [28]:
cal_sigma_star(11.5, 1.0, 0.5)

np.float64(1350707357.7031496)

In [34]:
file_path = './raw/observations_deV_with_mass_grids.hdf5'
overwrite_existing = False

updated = 0
skipped = 0

with h5py.File(file_path, 'r+') as deV_file:
    for name in deV_file:
        grp = deV_file[name]
        zd = grp.attrs['zd']
        re_arcsec = grp.attrs['reff_deV']
        logmstar = grp.attrs['logmchab_deV']
        log10_sigma_star = cal_log10_sigma_star(logmstar, re_arcsec, zd)

        # 仅新增 attrs，默认不覆盖已有值，避免误改原有结果
        if (not overwrite_existing) and ('log10_sigma_star' in grp.attrs):
            skipped += 1
            continue

        grp.attrs['log10_Sigma_star'] = float(log10_sigma_star)
        updated += 1

print(f'Write finished: updated={updated}, skipped={skipped}, file={file_path}')

Write finished: updated=23, skipped=0, file=./raw/observations_deV_with_mass_grids.hdf5


In [36]:
file_path = './raw/observations_with_mass_grids_all.hdf5'
overwrite_existing = False

updated = 0
skipped = 0

with h5py.File(file_path, 'r+') as file:
    for name in file:
        grp = file[name]
        zd = grp.attrs['zd']
        re_arcsec = grp.attrs['re_arcsec']
        logmstar = grp.attrs['logmchab']
        log10_sigma_star = cal_log10_sigma_star(logmstar, re_arcsec, zd)

        # 仅新增 attrs，默认不覆盖已有值，避免误改原有结果
        if (not overwrite_existing) and ('log10_sigma_star' in grp.attrs):  
            skipped += 1
            continue

        grp.attrs['log10_Sigma_star'] = float(log10_sigma_star)
        updated += 1

print(f'Write finished: updated={updated}, skipped={skipped}, file={file_path}')

Write finished: updated=23, skipped=0, file=./raw/observations_with_mass_grids_all.hdf5
